# Distillation RL-Assisted MPC Horizons

This notebook keeps the shared horizon runner, plotting, and data layout used by the unified project, while matching the archived distillation horizon notebooks for default setpoints, run lengths, observer poles, and horizon tuning settings. The comments are written to stay close to the legacy notebook tone so the experiment is easier to follow and modify.

In [ ]:
from pathlib import Path
import os

from utils.notebook_setup import prepare_distillation_notebook_env, print_notebook_summary

# Main notebook controls.
# These defaults match the archived distillation horizon notebooks unless you intentionally override them.
RUN_MODE = "nominal"  # "nominal" | "disturb"
DISTURBANCE_PROFILE = "none"  # "none" | "ramp" | "fluctuation"
STATE_MODE = "standard"  # "standard" | "mismatch"
STYLE_PROFILE = "hybrid"  # "hybrid" | "paper" | "debug"
SAVE_PDF = False

# Aspen selection. Leave ASPEN_PRESET="default" to follow the legacy family-specific plant file.
ASPEN_PRESET = "default"
ASPEN_PATH_OVERRIDE = None
SNAPS_PATH_OVERRIDE = None
ASPEN_ROOT_OVERRIDE = None
DISTILLATION_VISIBLE = True

# Canonical distillation folders. Leave as None to use Distillation/Data and Distillation/Results.
DISTILLATION_DATA_DIR_OVERRIDE = None
DISTILLATION_RESULTS_DIR_OVERRIDE = None

# Optional naming/path overrides.
RESULT_PREFIX_OVERRIDE = None
COMPARE_PREFIX_OVERRIDE = None
BASELINE_MPC_PATH_OVERRIDE = None

# Optional run-size overrides. Leave as None to keep the archived notebook defaults.
N_TESTS_OVERRIDE = None
SET_POINTS_LEN_OVERRIDE = None
WARM_START_OVERRIDE = None
TEST_CYCLE_OVERRIDE = None
PLOT_START_EPISODE_OVERRIDE = None
COMPARE_START_EPISODE_OVERRIDE = None

REPO_ROOT, DATA_DIR, RESULT_DIR, DISTURBANCE_PROFILE, DYN_PATH, SNAPS_PATH, ASPEN_SOURCE = prepare_distillation_notebook_env(
    run_mode=RUN_MODE,
    disturbance_profile=DISTURBANCE_PROFILE,
    family="horizon",
    aspen_preset=ASPEN_PRESET,
    dyn_path_override=ASPEN_PATH_OVERRIDE,
    snaps_path_override=SNAPS_PATH_OVERRIDE,
    aspen_root_override=ASPEN_ROOT_OVERRIDE,
    data_dir_override=DISTILLATION_DATA_DIR_OVERRIDE,
    results_dir_override=DISTILLATION_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)

print_notebook_summary(
    "Distillation horizon notebook configuration",
    {
        "Distillation data dir": DATA_DIR,
        "Distillation results": RESULT_DIR,
        "Run mode": RUN_MODE,
        "Disturbance profile": DISTURBANCE_PROFILE,
        "State mode": STATE_MODE,
        "Aspen source": ASPEN_SOURCE,
        "Aspen dynf": DYN_PATH,
        "Aspen snaps": SNAPS_PATH,
    },
)

import numpy as np
import torch

from DQN.dqn_agent import DQNAgent
from systems.distillation import DISTILLATION_SYSTEM_METADATA
from systems.distillation.config import (
    DELTA_T_HOURS,
    DISTILLATION_HORIZON_RUN_PROFILES,
    DISTILLATION_INPUT_BOUNDS,
    DISTILLATION_NOMINAL_CONDITIONS,
    DISTILLATION_OBSERVER_POLES,
    DISTILLATION_RL_SETPOINTS_PHYS,
    DISTILLATION_SETPOINT_RANGE_PHYS,
    DISTILLATION_SS_INPUTS,
    HORIZON_CONTROL_GRID,
    HORIZON_PREDICT_GRID,
    RL_REWARD_DEFAULTS,
)
from systems.distillation.data_io import canonical_baseline_path, load_distillation_system_data
from systems.distillation.plant import build_distillation_system, distillation_system_stepper
from systems.distillation.scenarios import build_distillation_disturbance_schedule
from Simulation.mpc import MpcSolverGeneral
from utils.helpers import apply_min_max, build_horizon_recipes
from utils.horizon_runner import run_dqn_mpc_horizon_supervisor
from utils.observer import compute_observer_gain
from utils.plotting import compare_mpc_rl_from_dirs, plot_horizon_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim


In [ ]:
RUN_PROFILE = DISTILLATION_HORIZON_RUN_PROFILES[(RUN_MODE, DISTURBANCE_PROFILE)]


nominal_conditions = DISTILLATION_NOMINAL_CONDITIONS.copy()
ss_inputs = DISTILLATION_SS_INPUTS.copy()
u_min = DISTILLATION_INPUT_BOUNDS["u_min"].copy()
u_max = DISTILLATION_INPUT_BOUNDS["u_max"].copy()
setpoint_y = DISTILLATION_SETPOINT_RANGE_PHYS.copy()

# Legacy horizon notebooks used the RL setpoint pair below.
y_sp_scenario_phys = DISTILLATION_RL_SETPOINTS_PHYS.copy()
system = build_distillation_system(
    path=DYN_PATH,
    ss_inputs=ss_inputs,
    initialization_point=nominal_conditions,
    delta_t=DELTA_T_HOURS,
    visible=DISTILLATION_VISIBLE,
)
steady_states = {"ss_inputs": np.asarray(system.ss_inputs, float).copy(), "y_ss": np.asarray(system.y_ss, float).copy()}
system_data = load_distillation_system_data(REPO_ROOT, steady_states, setpoint_y, u_min, u_max)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:])

RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or f"distillation_horizon_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}_unified"
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or f"distillation_compare_horizon_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}"
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else canonical_baseline_path(REPO_ROOT, RUN_MODE, DISTURBANCE_PROFILE)
n_tests = RUN_PROFILE["n_tests"] if N_TESTS_OVERRIDE is None else int(N_TESTS_OVERRIDE)
set_points_len = RUN_PROFILE["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else int(SET_POINTS_LEN_OVERRIDE)
warm_start = RUN_PROFILE["warm_start"] if WARM_START_OVERRIDE is None else int(WARM_START_OVERRIDE)
TEST_CYCLE = list(RUN_PROFILE["test_cycle"]) if TEST_CYCLE_OVERRIDE is None else list(TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = RUN_PROFILE.get("plot_start_episode", 1) if PLOT_START_EPISODE_OVERRIDE is None else int(PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = RUN_PROFILE.get("compare_start_episode", PLOT_START_EPISODE) if COMPARE_START_EPISODE_OVERRIDE is None else int(COMPARE_START_EPISODE_OVERRIDE)
TOTAL_STEPS = n_tests * set_points_len * len(y_sp_scenario_phys)
DISTURBANCE_SCHEDULE = build_distillation_disturbance_schedule(RUN_MODE, DISTURBANCE_PROFILE, TOTAL_STEPS)


print_notebook_summary(
    "Resolved experiment parameters",
    {
        "n_tests": n_tests,
        "set_points_len": set_points_len,
        "warm_start": warm_start,
        "plot_start_episode": PLOT_START_EPISODE,
        "compare_start_episode": COMPARE_START_EPISODE,
        "legacy_horizon_setpoints_phys": y_sp_scenario_phys.tolist(),
        "baseline_mpc_path": BASELINE_MPC_PATH,
    },
)


In [ ]:
# User-editable controller defaults. Edit these values directly when you want to change controller/agent behavior.
poles = DISTILLATION_OBSERVER_POLES.copy()
L = compute_observer_gain(A_aug, C_aug, poles)

PREDICT_GRID = HORIZON_PREDICT_GRID
CONTROL_GRID = HORIZON_CONTROL_GRID
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
STATE_DIM = get_rl_state_dim(A_aug.shape[0], C_aug.shape[0], B_aug.shape[1], STATE_MODE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HIDDEN_LAYERS = [512, 512, 512, 512, 512]
BUFFER_SIZE = 40_000
DECISION_INTERVAL = 4

predict_h = 6
cont_h = 3
Q1_penalty = 1.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0

reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    n_inputs=2,
    **RL_REWARD_DEFAULTS,
)

nominal_qi = 0.0
nominal_qs = 0.0
nominal_hA = 0.0
qi_change = 1.0
qs_change = 1.0
ha_change = 1.0


print(f"Using device: {DEVICE}")
print(f"State dimension: {STATE_DIM}")
print(f"Number of horizon actions: {len(HORIZON_RECIPES)}")
print(reward_params)

dqn_agent = DQNAgent(
    state_dim=STATE_DIM,
    action_dim=len(HORIZON_RECIPES),
    hidden_dim=HIDDEN_LAYERS,
    gamma=0.99,
    lr=1e-4,
    batch_size=128,
    buffer_size=BUFFER_SIZE,
    grad_clip_norm=10.0,
    double_dqn=True,
    target_update="soft",
    tau=0.01,
    hard_update_interval=10_000,
    target_combine="q1",
    activation="relu",
    use_layer_norm=False,
    dropout=0.0,
    device=DEVICE,
    eps_start=0.3,
    eps_end=0.01,
    eps_decay_rate=0.99999,
    eps_decay_mode="exp",
)


In [ ]:
horizon_cfg = {
    "mode": RUN_MODE,
    "state_mode": STATE_MODE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "decision_interval": DECISION_INTERVAL,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
}

runtime_ctx = {
    "system": system,
    "y_sp_scenario": y_sp_scenario,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "agent": dqn_agent,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "L": L,
    "data_min": data_min,
    "data_max": data_max,
    "horizon_recipes": HORIZON_RECIPES,
    "reward_fn": reward_fn,
    "system_metadata": DISTILLATION_SYSTEM_METADATA,
    "reward_params": reward_params,
    "system_stepper": distillation_system_stepper,
    "disturbance_labels": DISTILLATION_SYSTEM_METADATA.get("disturbance_labels"),
    "disturbance_schedule": DISTURBANCE_SCHEDULE,
}

result_bundle = run_dqn_mpc_horizon_supervisor(horizon_cfg=horizon_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH
result_bundle["reward_params"] = reward_params
result_bundle["run_profile"] = RUN_PROFILE

print(f"nFE: {result_bundle['nFE']}")
print(f"Avg reward samples: {len(result_bundle['avg_rewards'])}")


In [ ]:
out_dir_rl = plot_horizon_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "recipe_counts": True,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=BASELINE_MPC_PATH,
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_MODE,
    start_episode=COMPARE_START_EPISODE,
    save_pdf=SAVE_PDF,
    style_profile=STYLE_PROFILE,
)

print(out_dir_rl)
print(out_dir_cmp)

try:
    system.close(SNAPS_PATH)
except Exception:
    pass
